# 逻辑回归数学基础

这份 Notebook 用来整理进入逻辑回归之前最需要掌握的数学基础，重点放在：

- 概率与条件概率
- 伯努利分布与二分类标签
- odds 与 log-odds
- sigmoid 函数
- 从线性分数到分类概率
- 逻辑回归中的基本记号

目标不是一开始就把所有推导做完，而是先搭好一条你后面学习逻辑回归时会一直用到的数学主线。

## 1. 为什么要先补这部分数学基础

在线性回归中，我们预测的是一个连续值：

$$
\hat{y} = Xw + b
$$

但在逻辑回归中，我们通常面对的是二分类问题，例如：

- 这封邮件是不是垃圾邮件
- 这个病人是否患病
- 这名学生能否通过考试

这时标签不再是任意实数，而通常写成：

$$
y \in \{0, 1\}
$$

于是问题就变成了：给定特征 $x$，我们如何预测样本属于 1 类的概率？

也就是：

$$
P(y = 1 \mid x)
$$

所以逻辑回归真正的数学入口不是先看代码，而是先理解概率、条件概率、概率分布，以及概率是怎样和线性模型连起来的。

## 2. 基本记号

为了和你前面的线性回归笔记保持一致，我们仍然使用下面这些记号：

$$
X \in \mathbb{R}^{m \times n}
$$

表示特征矩阵，其中：

- $m$ 是样本数
- $n$ 是特征数

第 $i$ 个样本记作：

$$
x^{(i)} \in \mathbb{R}^{n}
$$

标签向量记作：

$$
y \in \{0,1\}^{m}
$$

也就是说，在逻辑回归里，标签向量中的每个元素都只有两种可能：0 或 1。

参数仍然记作：

$$
w \in \mathbb{R}^{n}, \qquad b \in \mathbb{R}
$$

线性部分仍然是：

$$
z = w^T x + b
$$

后面逻辑回归的大部分新内容，都是围绕这个线性分数 $z$ 如何变成概率来展开的。

## 3. 概率的最基本含义

概率可以粗略理解成一件事情发生的可能性大小。

如果事件 $A$ 发生的概率记作：

$$
P(A)
$$

那么它满足：

$$
0 \le P(A) \le 1
$$

其中：

- $P(A)=0$ 表示几乎不可能发生
- $P(A)=1$ 表示一定发生
- $P(A)=0.5$ 表示发生与不发生同样可能

在逻辑回归里，我们最关心的概率通常不是任意事件的概率，而是“样本属于正类的概率”。

例如：

$$
P(y=1 \mid x)
$$

它表示：在样本特征为 $x$ 的条件下，这个样本属于 1 类的概率。

## 4. 条件概率

条件概率是逻辑回归里最核心的概率概念之一。

它的形式是：

$$
P(A \mid B)
$$

意思是：在事件 $B$ 已经发生的条件下，事件 $A$ 发生的概率。

在机器学习里，通常可以这样理解：

- $B$ 是“给定输入特征 $x$”
- $A$ 是“标签属于某一类”

所以逻辑回归学习的不是：

$$
P(y=1)
$$

而是：

$$
P(y=1 \mid x)
$$

这里的差别很重要：

- $P(y=1)$ 只看整体正类比例
- $P(y=1 \mid x)$ 会根据样本自己的特征来改变

机器学习模型真正想做的，就是输入不同的 $x$，输出不同的条件概率。

## 5. 随机变量与伯努利分布

逻辑回归里的标签只有两种取值：0 和 1。

这种只有两种结果的随机变量，很适合用伯努利分布来描述。

如果随机变量 $Y$ 满足：

$$
Y \in \{0,1\}
$$

并且：

$$
P(Y=1)=p, \qquad P(Y=0)=1-p
$$

那么就说 $Y$ 服从参数为 $p$ 的伯努利分布。

记作：

$$
Y \sim \mathrm{Bernoulli}(p)
$$

这和逻辑回归非常匹配，因为逻辑回归的输出正是一个概率 $p$，它表示：

$$
p = P(Y=1 \mid x)
$$

于是我们也就自动得到：

$$
P(Y=0 \mid x)=1-p
$$

这就是二分类概率建模最基础的一层。

## 6. 伯努利分布为什么能写成一个统一公式

对于单个样本，如果它的真实标签 $y$ 只能取 0 或 1，那么：

- 当 $y=1$ 时，我们希望对应的概率是 $p$
- 当 $y=0$ 时，我们希望对应的概率是 $1-p$

这两种情况可以统一写成：

$$
P(Y=y \mid x) = p^y (1-p)^{1-y}
$$

你可以自己代入验证：

当 $y=1$ 时：

$$
p^1(1-p)^0 = p
$$

当 $y=0$ 时：

$$
p^0(1-p)^1 = 1-p
$$

这个统一公式后面会直接连到逻辑回归的似然函数与交叉熵损失，所以值得先记住。

## 7. odds 与 log-odds

除了直接讨论概率 $p$，逻辑回归里还有两个特别重要的量：odds 和 log-odds。

### 7.1 odds

如果一个事件发生的概率是 $p$，不发生的概率是 $1-p$，那么它的 odds 定义为：

$$
\mathrm{odds} = \frac{p}{1-p}
$$

它表示“发生”和“不发生”的相对比值。

例如：

- 如果 $p=0.5$，那么 odds = 1
- 如果 $p=0.8$，那么 odds = 4
- 如果 $p=0.2$，那么 odds = 0.25

### 7.2 log-odds

再对 odds 取对数，就得到 log-odds：

$$
\log \frac{p}{1-p}
$$

它的重要性在于：

- 概率 $p$ 被限制在 $(0,1)$
- odds 被限制在 $(0,+\infty)$
- log-odds 则可以取任意实数

而线性模型 $w^T x+b$ 恰好也是任意实数。

于是，逻辑回归的核心思想之一就是：

$$
\log \frac{p}{1-p} = w^T x + b
$$

这句话非常关键，因为它把“概率”和“线性模型”真正连接起来了。

## 8. 从 log-odds 推出 sigmoid

如果我们设：

$$
z = w^T x + b
$$

并令：

$$
\log \frac{p}{1-p} = z
$$

那么就可以一步步解出 $p$。

先对两边取指数：

$$
\frac{p}{1-p} = e^z
$$

然后整理：

$$
p = e^z(1-p)
$$

$$
p = e^z - e^z p
$$

$$
p(1+e^z)=e^z
$$

$$
p = \frac{e^z}{1+e^z} = \frac{1}{1+e^{-z}}
$$

这就是 sigmoid 函数：

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

也就是说，逻辑回归不是随便选了一个函数，而是从 log-odds 的线性假设自然推出了 sigmoid。

## 9. sigmoid 函数的性质

sigmoid 函数：

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

有几个特别重要的性质：

### 9.1 值域在 0 到 1 之间

$$
0 < \sigma(z) < 1
$$

这正好适合拿来表示概率。

### 9.2 当 $z=0$ 时，输出为 0.5

$$
\sigma(0)=0.5
$$

这意味着线性分数为 0 时，模型认为两类一样可能。

### 9.3 当 $z$ 越大时，输出越接近 1

如果 $z \to +\infty$，那么：

$$
\sigma(z) \to 1
$$

### 9.4 当 $z$ 越小时，输出越接近 0

如果 $z \to -\infty$，那么：

$$
\sigma(z) \to 0
$$

### 9.5 sigmoid 是单调递增的

这表示：

- 线性分数更大
- 属于正类的概率也更大

这让模型的解释很自然。

## 10. 从线性分数到分类概率

现在可以把逻辑回归最核心的预测过程写出来：

第一步，先算线性分数：

$$
z = w^T x + b
$$

第二步，再用 sigmoid 把它变成概率：

$$
p = \sigma(z) = \frac{1}{1+e^{-z}}
$$

并把这个概率解释成：

$$
p = P(y=1 \mid x)
$$

于是也就有：

$$
P(y=0 \mid x)=1-p
$$

这比线性回归多出来的一步，就是逻辑回归的核心变化。

你可以把它简单记成：

线性回归：

$$
\hat{y} = w^T x + b
$$

逻辑回归：

$$
\hat{p} = \sigma(w^T x + b)
$$

前者输出的是连续值，后者输出的是概率。

## 11. 阈值与决策边界

逻辑回归先输出概率，但最终分类时通常还要给出一个类别标签。

最常见的规则是使用 0.5 作为阈值：

- 如果 $P(y=1 \mid x) \ge 0.5$，预测为 1
- 如果 $P(y=1 \mid x) < 0.5$，预测为 0

由于：

$$
\sigma(0)=0.5
$$

所以分界点满足：

$$
w^T x + b = 0
$$

这说明一个很重要的事实：虽然逻辑回归输出的是非线性的 sigmoid 概率，但它的决策边界仍然是线性的。

在二维平面中，它通常是一条直线；在更高维空间里，它是一个超平面。

## 12. 为什么这会引出交叉熵

有了概率模型以后，我们就会自然地问：如果模型给出了一个概率 $p$，怎样衡量这个概率预测得好不好？

在线性回归里，我们常用平方误差：

$$
(y-\hat{y})^2
$$

但在逻辑回归中，预测对象已经变成了概率，而且标签服从伯努利分布，于是更自然的做法是从似然出发，得到对数损失，也就是后面会学到的二元交叉熵：

$$
-\left[y\log p + (1-y)\log(1-p)\right]
$$

这部分不用在本 Notebook 里完全展开，你现在只需要先建立这条联系：

- 二分类标签
- 伯努利分布
- 条件概率 $P(y=1\mid x)$
- sigmoid 输出概率
- 进一步引出交叉熵损失

这就是逻辑回归的数学主线。

## 13. 进入正式学习前，你应该已经能回答的问题

如果这份基础笔记读顺了，那么你应该已经能回答下面这些问题：

1. 为什么逻辑回归适合做二分类？
2. $P(y=1\mid x)$ 和 $P(y=1)$ 的区别是什么？
3. 伯努利分布为什么适合描述逻辑回归中的标签？
4. odds 和 log-odds 是什么？
5. 为什么线性分数 $w^Tx+b$ 可以和 log-odds 联系起来？
6. sigmoid 函数为什么能把任意实数映射到 $(0,1)$？
7. 为什么逻辑回归的决策边界仍然是线性的？

如果这些点都能说清楚，后面再去学损失函数、梯度下降和 sklearn 实战就会顺很多。

## 14. 本节小结

可以把这份 Notebook 压缩成下面这条主线：

$$
y \in \{0,1\}
$$

$$
P(y=1\mid x)=p
$$

$$
\log\frac{p}{1-p}=w^Tx+b
$$

$$
p=\sigma(w^Tx+b)=\frac{1}{1+e^{-(w^Tx+b)}}
$$

这就是逻辑回归最核心的数学起点。

下一步通常就可以继续学习：

- 二元交叉熵
- 逻辑回归的梯度
- 梯度下降如何训练参数
- sklearn 中的 `LogisticRegression`

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_values = np.array([-3, -1, 0, 1, 3])
p_values = sigmoid(z_values)

for z, p in zip(z_values, p_values):
    print(f"z = {z:>2}, sigmoid(z) = {p:.6f}")
